# ЛИНЕЙКА ЖЕСТКАЯ

## IMPORT

In [1]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator,MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml.feature import VectorAssembler, OneHotEncoder

from pyspark.sql.functions import col,when,median
from pyspark.sql import SparkSession, Window
import pyspark.sql.functions as F

import numpy as np
import math
import tqdm
import itertools
import json
import os
import shutil

os.environ['HADOOP_HOME'] = "C:\\hadoop" 
# Добавляем путь к bin в системный PATH
os.environ['PATH'] += os.pathsep + "C:\\hadoop\\bin"

## СОЗДАНИЕ СЕССИИ

In [2]:
spark = SparkSession.builder.appName('Logistic_reg')\
    .config("spark.driver.memory", "5g") \
    .config("spark.executor.memory", "5g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

## ФУНКЦИЯ ПО СБОРКЕ ***КЕТА

In [3]:
def finalize_csv(folder_path, target_name):
    # 1. Проверяем, существует ли папка
    if not os.path.exists(folder_path):
        print(f"Папка {folder_path} не найдена")
        return

    files = os.listdir(folder_path)
    
    # 2. Ищем файл данных (обычно начинается на part- и заканчивается на .csv)
    # Важно: Spark создает скрытые файлы ._SUCCESS и .crc, их игнорируем
    part_files = [f for f in files if f.startswith("part-") and f.endswith(".csv")]
    
    if not part_files:
        print("Файл с данными не найден в папке")
        return
        
    part_file = part_files[0]
    
    # 3. Формируем путь назначения (в ту же папку, где лежала папка-результат)
    parent_dir = os.path.dirname(os.path.abspath(folder_path))
    final_path = os.path.join(parent_dir, target_name)
    
    # 4. Переносим файл (shutil.move надежнее os.rename между разделами/дисками)
    temp_full_path = os.path.join(folder_path, part_file)
    
    if os.path.exists(final_path):
        os.remove(final_path)
        
    shutil.move(temp_full_path, final_path)
    
    # 5. Удаляем всю временную папку целиком (вместе со скрытыми файлами .crc)
    shutil.rmtree(folder_path)
    print(f"Готово! Файл сохранен как: {final_path}")

## LOAD DOTA

In [4]:
# Load training data
train_data_path = '../datasets/joined/train_data.parquet'
valid_data_path = '../datasets/joined/valid_data.parquet'
features_path = '../datasets/joined/columns_list.json'
# Load train data
training = spark.read.parquet(train_data_path)
# Load valid data
# validating = spark.read.parquet(valid_data_path)

## СОЗДАНИЕ СПИСКА ПРИЗНАКОВ

In [5]:
with open(features_path, 'r') as file:
        cols = json.load(file)

# СОЗДАЛ КОЛОНКИ
feature_cols = cols['all']
cat_cols = cols['cat']
numeric_cols = list(set(feature_cols) - set(cat_cols))

## КОСТЫЛЬНО-МЕДИЦИНСКИЕ ВМЕШАТЕЛЬСТВА В ДАННЫЕ

In [6]:

for cat in cat_cols:
        training = training.withColumn(cat,col(cat)+1)

# ВХОДНЫЕ КАТЕГОРИАЛЬНЫЕ КОЛОНКИ
to_encode_in = cat_cols
to_encode_out = [c+"_vec" for c in cat_cols]


## ПРЕПРОЦЕСС + КРОСВАЛ + ГРИДСЕРЕГА

In [7]:

# ОБЪЯВИЛ ЭНКОДЕР И ОБРАБОТАЛ
encoder = OneHotEncoder(inputCols=to_encode_in,
                        outputCols=to_encode_out,
                        handleInvalid='keep')  # Вот это спасет от ошибки)
training_encoded = encoder.fit(training).transform(training)

# ВЕКТОРИЗОВАЛ
assembler_inputs = to_encode_out + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

final_training = assembler.transform(training_encoded) 

# # Define the model
# lr = LogisticRegression(
#     featuresCol='features',
#     labelCol='target'
# )

# # Create parameter grid
# paramGrid = (ParamGridBuilder()
#              .addGrid(lr.regParam, [0,0.01, 0.1,])
#              .addGrid(lr.elasticNetParam, [0.01,0.04])
#              .addGrid(lr.maxIter, [50])
#              .build())

# # Define evaluator
# evaluator = BinaryClassificationEvaluator(
#     rawPredictionCol='rawPrediction',
#     labelCol='target',
#     metricName='areaUnderROC'
# )

# sample_fraction = 0.2
# print(f"Sampling {sample_fraction*100}% of data for grid search...")
# training_sample = final_training.sample(False, sample_fraction, seed=42)

# # ЗДЕСЬ ЖЕСТКО НЕ УВЕРЕН ЧТО ЭТО НАДО ВООБЩЕ
# training_sample.cache()
# print(f"Training sample count: {training_sample.count()}")

# CrossValidator with sampling for big data
# For production, you might want to use 3-5 folds
# crossval = CrossValidator(
#     estimator=lr,
#     estimatorParamMaps=paramGrid,
#     evaluator=evaluator,
#     numFolds=3,
#     parallelism=2,  # Adjust based on your cluster
#     seed=42
# )

# print("Starting grid search...")
# cvModel = crossval.fit(training_sample)

# best_lr_model = cvModel.bestModel
# best_reg_param = best_lr_model.getRegParam()
# best_elastic_net = best_lr_model.getElasticNetParam()
# best_max_iter = best_lr_model.getMaxIter()
# sample_predictions = best_lr_model.transform(training_sample)
# sample_auc = evaluator.evaluate(sample_predictions)
# print(f"Sample AUC: {sample_auc:.4f}")

# training_sample.unpersist()

## ВЫВОД ПАРАМЕТРОВ

In [8]:
# print(f'ALPHA: {best_reg_param}, L2RATIO: {best_elastic_net}, MAXITER: {best_max_iter}')

## ФИТ ЛУЧШЕЙ МОДЕЛИ

In [9]:
# Create new model with best parameters
best_lr = LogisticRegression(
    featuresCol='features',
    labelCol='target',
    regParam=0,
    elasticNetParam=0.1,
    maxIter=50
)

# Train on FULL dataset
final_model = best_lr.fit(final_training)
print("Full model training complete!")

Full model training complete!


## ВАЛИДАЦИОННЫЕ ИСПЫТАНИЯ

### ЗАГРУЖАЕМ ВАЛИД

In [10]:
# Load train data
validating = spark.read.parquet(train_data_path)

In [11]:
for cat in cat_cols:
        validating = validating.withColumn(cat,col(cat)+1)
validating_encoded = encoder.fit(training).transform(validating)
final_validating = assembler.transform(validating_encoded) 

### ПОИСКИ ЛУЧШЕЙ ЖИЗНИ

In [12]:
# 1. Трансформируем валидационные данные
pred_valid = final_model.transform(final_validating)
# Конвертируем сырые предсказания в колонку
pred_valid = pred_valid.withColumn("raw_arr", vector_to_array("rawPrediction"))
pred_valid = pred_valid.withColumn("raw_1", F.col("raw_arr")[1])

In [13]:
# Создаём окно
time_col = F.col("event_dttm").cast("long")
hours_back = 24
seconds_in_hour = 3600
window_24h = Window.partitionBy("customer_id") \
                          .orderBy(time_col) \
                          .rangeBetween(-hours_back * seconds_in_hour, -1)


In [14]:
eval_general = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction")

In [ ]:
# 1. Готовим ассемблер заранее
assembler = VectorAssembler(inputCols=["ultima_answer_raw"], outputCol="features_for_mini")

# 2. Сетка по перебору операций и параметров

methods_arr=['mean','max','stddev','median']
coef_arr = np.arange(-0.2,0.21,0.2)

mini_lr = LogisticRegression(
    featuresCol='features_for_mini',
    labelCol='target',
)
results = {}

for method in methods_arr:

    # Исправление для медианы
    if method == 'median':
        func = lambda c: F.percentile_approx(c, 0.5)
    else:
        func = getattr(F, method)

    current_method_df  = pred_valid.withColumn("max_attention_24h", func("raw_1").over(window_24h))

    for coef in coef_arr:

        loop_df = current_method_df.withColumn("ultima_answer_raw", coef * col("max_attention_24h")+col("raw_1")).na.fill(0, ["ultima_answer_raw"]) # Заменяем NULL на 0
        # Создаём вектор для мини линейки
        loop_df_vec = assembler.transform(loop_df)
        loop_df_vec = loop_df_vec.drop('prediction','rawPrediction','probability')
        mini_lr_fitted = mini_lr.fit(loop_df_vec)
        pred_ans_valid = mini_lr_fitted.transform(loop_df_vec)

        # 4. Считаем метрики (указываем нужную метрику через setMetricName)
        acc = eval_general.setMetricName("accuracy").evaluate(pred_ans_valid)
        prec = eval_general.setMetricName("weightedPrecision").evaluate(pred_ans_valid)
        rec = eval_general.setMetricName("weightedRecall").evaluate(pred_ans_valid)
        f1 = eval_general.setMetricName("f1").evaluate(pred_ans_valid)

        results[(method,coef)] = [acc, prec, rec,f1]
        
        print(f'koef {coef} done.')
    print(f'method {method} done.')


In [ ]:
# Превращаем словарь в список кортежей: ((method, coef), [metrics...])
sorted_results = sorted(
    results.items(), 
    key=lambda x: x[1][3], # x[1] — это список метрик, индекс 3 — это F1
    reverse=True           # От большего к меньшему
)

# Красивый вывод топ-10 результатов
print(f"{'Method':<10} | {'Coef':<10} | {'Precision':<10} | {'F1-Score':<10}")
print("-" * 50)

for (method, coef), metrics in sorted_results[:10]:
    acc, prec, rec, f1 = metrics
    print(f"{method:<10} | {coef:<10.2f} | {prec:<10.4f} | {f1:<10.4f}")

## ЗАКАЧКА ТЕСТА

In [ ]:
#  Load test data
test_data_path = '../datasets/joined/test_data.parquet'
testing = spark.read.parquet(test_data_path)

testing = testing.withColumn("was_hacked", 
    when(col("was_hacked") >0, 1).cast("byte")       # В остальных случаях пытаемся в число
).fillna(-1, subset=["was_hacked"])                  # Все пустые/невалидные -> -1
for cat in cat_cols:
        testing = testing.withColumn(cat,col(cat)+1)
        
# ОБЪЯВИЛ ЭНКОДЕР И ОБРАБОТАЛ
test_encoded = encoder.fit(training).transform(testing)

final_test = assembler.transform(test_encoded) 



## ЗДЕСЬ ТАНЦЫ С БУБНОМ (АНСАМБЛЬ И ПРОЧИЕ ПРИБЛУДЫ)

### 1. ДЕЛАЕМ ПРЕДСКАЗАНИЯ

In [ ]:
# 1. Трансформируем тестовые данные
predictions = final_model.transform(final_test)

### 2. РАСПРОСТРАНЯЕМ ЧЕРЕЗ ОКОНКУ НА Х ЧАСОВ ПО ЮЗЕРУ

In [ ]:
# 1. Сначала превращаем вектор предсказаний в массив, чтобы можно было считать числа
predictions = predictions.withColumn("raw_arr", vector_to_array("rawPrediction"))
# Берем значение для класса 1 (фрод)
predictions = predictions.withColumn("raw_1", F.col("raw_arr")[1])

In [ ]:
time_col = F.col("event_dttm").cast("long")
hours_back = 24
seconds_in_hour = 3600
window_24h = Window.partitionBy("customer_id") \
                          .orderBy(time_col) \
                          .rangeBetween(-hours_back * seconds_in_hour, -1)

# 1. Считаем количество операций (самый важный признак для фрода "набегами")
predictions = predictions.withColumn("max_attention_24h", F.stddev("raw_1").over(window_24h))
        

In [ ]:
# Заполняем null (для первой операции юзера, где окна еще нет)
predictions = predictions.fillna(0, subset=["max_attention_24h"])

### 3. ДЕЛАЕМ КОММИТ () ИЛИ СЛОЖИТЬ ИЛИ ОБУЧИТЬ НОВУЮ ЛИНЕЙКУ

In [ ]:
predictions = predictions.withColumn("ultima_answer", 0.2 * col("max_attention_24h")+col("raw_1"))

## ДУРАЦКАЯ РАБОТА С ПАПКАМИ

In [ ]:

# 2. Достаем вероятность КЛАССА 1 (фрод) из вектора [p0, p1]
# vector_to_array превращает вектор в массив, [1] берет второй элемент
# y_out = predictions.withColumn("prob_array", vector_to_array("probability")) \
#                    .select(
#                        F.col("event_id"), 
#                        F.col("prob_array")[1].alias("predict")
#                    )

y_out = predictions.select(
                       F.col("event_id"), 
                       F.col("ultima_answer").alias("predict")
                   )

# 3. Сохраняем во временную ПАПКУ
temp_path = '../submit/temp_folder'
y_out.coalesce(1).write.csv(
    temp_path,
    header=True,
    mode='overwrite'
)

# 4. Вызываем твою функцию 
# Первый аргумент - ПАПКА, которую создал Spark
# Второй аргумент - ИМЯ итогового файла
finalize_csv(temp_path, "lr_submit_1.csv")


Готово! Файл сохранен как: e:\antifraud_hak\submit\lr_submit_1.csv
